# Install Required Software with Pixi

<br>
    
This notebook installs the Pixi software environment required to run the other notebooks in the NISAR GCOV Cookbook.

If you launched this Cookbook in JupyterLab from the command line using `pixi run lab`, you already have the environment and do not need to execute this notebook 

<hr>

## Overview

1. [Prerequisites](env-prereqs)
1. [(Option 1) Install the environment with the Pixi pacakage manager](env-pixi)
1. [Summary](env-summary)
1. [Resources and references](env-resources)

<hr>

(env-prereqs)=
## 1. Prerequisites

| Prerequisite | Importance | Notes |
| --- | --- | --- |
| [The Pixi package manager must be installed on the system running Jupyter Lab](https://pixi.sh/latest/installation/) | Necessary | Pixi may not be supported on all Jupyter Hubs|
| [UW Scientific Software Engineering Center's Pixi guide](https://rse-guidelines.readthedocs.io/en/latest/fundamentals/computing-development-environments/pixi/) | Helpful | This is a great resource to get quickly get started with Pixi|

- **Rough Notebook Time Estimate**: 3 minutes

<hr>

(env-pixi)=
## 2. Install the environment with the Pixi pacakage manager

### 2a. Create a `work_dir` context manager

This is useful when you want to run a command in a working directory and then automatically change back to your original directory 

In [1]:
import os
from contextlib import contextmanager
from pathlib import Path

@contextmanager
def work_dir(new_dir):
    old_dir = os.getcwd()
    os.chdir(new_dir)
    try:
        yield
    finally:
        os.chdir(old_dir)

### 2b. Install the `isce3` environment with Pixi

In [2]:
with work_dir(Path.cwd().parent):
    !pixi install -e isce3

⠁ solving              [────────────────────]  0/1                                                                                      
⠁ solving              [────────────────────]  0/1  queued: isce3@linux-64 (linu                                                
⠁ solving              [────────────────────]  0/1  queued: isce3@linux-64 (linu                                                
⠁ solving              [────────────────────]  0/2  queued: isce3@osx-arm64 (osx                                                
⠁ solving              [────────────────────]  0/3  queued: isce3@osx-arm64 (osx                                                
⠁ solving              [────────────────────]  0/3  queued: isce3@osx-arm64 (osx                                                
⠁ solving              [────────────────────]  0/3  queued: isce3@osx-arm64 (osx                                                
⠉ solving              [────────────────────]  0/3  queued: isce3@osx-arm64 (osx         

### 2c. Register the `isce3` environment's Python kernel with `ipykernel`

In [3]:
env_name = "isce3"
display_name = f'"{env_name} (Python)"'

!pixi run -e isce3 python -m ipykernel install \
  --user \
  --name $env_name \
  --display-name $display_name

Installed kernelspec isce3 in /home/jovyan/.local/share/jupyter/kernels/isce3            


### 2d. Ensure notebook shell commands run in the `isce3` environment

This ensures that shell commands executed from inside a notebook with `!` run in the notebook kernel’s environment. This works by launching the entire Jupyter kernel process inside the Pixi environment, so the kernel’s PATH, environment variables, and Python executable all come from that Pixi environment, not from the parent JupyterLab environment.

In [4]:
from jupyter_client.kernelspec import KernelSpecManager
import json

ksm = KernelSpecManager()
spec = ksm.get_kernel_spec(env_name)
kernel_dir = Path(spec.resource_dir)
kernel_json = kernel_dir / "kernel.json"

data = json.loads(kernel_json.read_text())
orig_argv = data.get("argv", [])

new_argv = [
    "pixi",
    "run",
    "--manifest-path",
    str(Path.cwd().parent),
    "-e",
    env_name,
] + orig_argv

data["argv"] = new_argv
kernel_json.write_text(json.dumps(data, indent=2))
print(f"Updated kernel.json at {kernel_json} with Pixi wrapper.")
print("argv:", data["argv"])

Updated kernel.json at /home/jovyan/.local/share/jupyter/kernels/isce3/kernel.json with Pixi wrapper.
argv: ['pixi', 'run', '--manifest-path', '/home/jovyan/NISAR_Cookbook', '-e', 'isce3', '/home/jovyan/NISAR_Cookbook/.pixi/envs/isce3/bin/python', '-Xfrozen_modules=off', '-m', 'ipykernel_launcher', '-f', '{connection_file}']


### 2e. (Optional) Delete the environment and remove its `kernelspec`

In [ ]:
# Uncomment and run the code below to delete your pixi environment

# with work_dir(Path.cwd().parent):
#     !pixi clean
#     !jupyter kernelspec remove isce3 -y

<hr>

(env-summary)=
## 3. Summary
Now that you have installed the software environment with Pixi, [make sure you have access to the data](https://github.com/ASFOpenSARlab/NISAR_GCOV_Cookbook/blob/main/notebooks/set_up_Earthdata_Login.md). You will then be ready to run the remaining notebooks in the NISAR GCOV Cookbook.
<hr>

(env-resources)=
## 4. Resources and references

### References
- [{abbr}`UW SSEC (Univertsity of Washington Scientific Software Engineering Center)`](https://escience.washington.edu/software-engineering/ssec/)

**Author:** Alex Lewandowski